# Passo 6 – Regra de operação (Fase B: aqui entra o custo)

A RF da Fase A está **congelada** (candle, janela, re-treino, variáveis e hiperparâmetros escolhidos só pelo R²). Agora medimos o **dinheiro** e ajustamos **apenas a regra de entrada**.

Cada candle: compra (+1) ou vende (−1) no fechamento de *t*, zera no fechamento de *t+1*, ou fica de fora (0).
$$\text{resultado} = \text{direção} \times (\text{close}_{t+1} - \text{close}_t) - 5 \text{ pts}$$

| Regra | Opera quando |
|---|---|
| sempre | todo candle, na direção prevista |
| previsto > k × custo | \|movimento previsto em pts\| > k × 5 (k = 1, 2, 3) |
| top X% | \|previsão\| acima do quantil (1 − X) das previsões do **trimestre anterior** |
| aleatório (controle) | mesmos candles da regra, direção sorteada 1.000 vezes |

**Regra de escolha (definida antes de rodar):** candidatas = esperança > 0 e positiva em ≥ 3 dos 4 trimestres de 2024; entre elas, a de **maior lucro total**. Se nenhuma for candidata, registramos que nenhuma regra torna o modelo lucrativo na validação.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
import rf_zero as rz
cfg = json.load(open("saidas/modelo_congelado.json")); print(cfg)
win_1m = rz.carregar("../../dados/WINN_M1/WIN$N_M1.csv")
base, cols = rz.montar_base(win_1m, cfg["candle"])
# começa em 10/2023: o 4º trimestre de 2023 só calibra o limiar "top X%" de jan/2024
d = rz.walk_forward(base, cols, ("2023-10-01", rz.VALIDACAO[1]), cfg["janela"], rz.FREQ_CODIGO[cfg["retreino"]],
                    fabrica=rz.fabrica_congelada(cfg))
d["prev_pts"] = rz.prev_pontos(d)
d.to_pickle("saidas/B6_previsoes.pkl")
v = d[d.index >= rz.VALIDACAO[0]]
print(len(v), "candles em 2024 | |previsto| mediano:", round(v["prev_pts"].abs().median(), 1), "pts")
rz.resumo_acerto(v).round(3)

In [ ]:
def direcoes(regra, valor=None):
    s = np.sign(v["prev_pts"])
    if regra == "sempre":
        return s
    if regra == "custo":
        return s.where(v["prev_pts"].abs() > valor * rz.CUSTO_PTS, 0)
    tri = d.index.to_period("Q")
    q = d["prev_pts"].abs().groupby(tri).quantile(1 - valor)
    lim = v.index.to_period("Q").map(lambda t: q[t - 1])
    return s.where(v["prev_pts"].abs().values > lim.values, 0)

REGRAS = [("sempre", None, "opera sempre")] + [("custo", k, f"previsto > {k}× custo") for k in [1, 2, 3]] + \
         [("top", x, f"{int(x*100)}% mais fortes") for x in [0.5, 0.3, 0.2, 0.1]]
rng = np.random.default_rng(42)
linhas, res_por_regra = {}, {}
for regra, valor, rot in REGRAS:
    dirc = direcoes(regra, valor); op = (dirc != 0).values
    res = pd.Series(rz.resultado_operacoes(dirc.values, v["mov_prox"].values), index=v.index[op])
    r = rz.resumo_ops(res.values, res.index)
    mov = v["mov_prox"].values[op]
    sims = (rng.choice([-1, 1], size=(1000, len(mov))) * mov).mean(axis=1) - rz.CUSTO_PTS
    r["p vs aleatório"] = (sims >= res.mean()).mean()
    r["ganho bruto pts/op"] = res.mean() + rz.CUSTO_PTS
    linhas[rot] = r; res_por_regra[rot] = res
tab = pd.DataFrame(linhas).T
tab.to_csv("saidas/B6_regras.csv")
cand = tab[(tab["esperança pts/op"] > 0) & (tab["trimestres +"] >= 3)]
REGRA = cand["total pts"].idxmax() if len(cand) else None
print("Candidatas:", list(cand.index), "| Escolha pela regra →", REGRA)
tab.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for rot in ["opera sempre", "previsto > 1× custo", "previsto > 2× custo", "20% mais fortes"]:
    r = res_por_regra[rot]
    (r.cumsum() * rz.REAIS_POR_PONTO["WIN"]).plot(ax=ax, label=f"{rot} ({len(r)} ops)")
ax.axhline(0, color="gray", lw=.8); ax.set_ylabel("R$ (1 contrato)"); ax.legend(fontsize=8)
ax.set_title("Capital acumulado em 2024, com custo de 5 pts")
plt.tight_layout(); plt.savefig("passo06_curvas_capital.png", dpi=120); plt.show()

In [ ]:
pd.DataFrame({k: r.groupby(r.index.quarter).mean() for k, r in res_por_regra.items()}).T.round(1).rename(columns=lambda q: f"T{q}")

In [ ]:
# sensibilidade ao custo (esperança por operação)
pd.DataFrame({rot: {f"custo {c}": (res_por_regra[rot] + rz.CUSTO_PTS).mean() - c for c in [0, 2, 5, 10, 15]}
              for rot in res_por_regra}).T.round(2)

In [ ]:
json.dump({**cfg, "regra": REGRA}, open("saidas/modelo_congelado.json", "w"), indent=1)

## Conclusões do Passo 6

*(preenchidas após rodar)*